# 16. SB-SPS, re-evaluation and pre-emption: MoReV2X against WiLabV2Xsim

How the two simulators implement the three Mode 2 mechanisms, step by step against the
standard, with the line of code behind every statement. The reference is TS 38.214 §8.1.4 (sensing
and selection, re-evaluation and pre-emption checks) and TS 38.321 §5.22.1 (SPS, reselection
counter, which resources are replaced).

* **MoReV2X**: this repository, branch `feature/qos-evaluation`, `src/MoReV2X/model/nr-v2x-ue-mac.cc` (the fork's priorities,
  pre-emption and multi-flow are included).
* **WiLabV2Xsim**: `main` at 612bf26 with `wilabv2xsim/patches/wilabv2xsim-octave-dit.patch`
  applied (`wilabv2xsim/setup.sh` rebuilds it), so line numbers are those of the patched files.
  The patch adds the Octave port and DIT with its pre-emption check; everything else is upstream.

Notebook 15 used WiLabV2Xsim to reproduce Todisco, Wu and Bazzi (IEEE VNC 2024); notebooks 12 to
14 used MoReV2X. The differences below explain why their numbers cannot be compared one to one.

In [ ]:
import os, sys, json, subprocess
sys.path.insert(0, os.path.abspath("../../tools"))
sys.path.insert(0, os.path.abspath("../../tools/qos"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

mr.apply_chart_style()
os.chdir(mr.ROOT)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

from matplotlib.patches import Rectangle

## 1. What a resource is

| | WiLabV2Xsim | MoReV2X |
|---|---|---|
| Unit of allocation | a **beacon resource (BR)**: a fixed block of subchannels in one TTI, the same size for every packet of the run. The grid is `NbeaconsT` TTIs × `NbeaconsF` BRs per TTI, repeated every allocation period | a **candidate single-slot resource (CSR)**: a start subchannel and a length `L_SubCh` in one slot, sized for each packet |
| Packet sizes | one size for the whole run (`beaconSizeBytes`) | any size up to one slot; a larger packet triggers a size reselection |
| Reservation periods | one: the allocation period (`allocationPeriod`) for every UE | per flow (RRI of the class); several RRIs coexist |
| Priorities | none upstream | SL priority 1–8 per flow, carried in the SCI |
| Sensed information | per UE: `sensingMatrixCV2X` (power received on each BR) and `knownUsedMatrixCV2X` (BR announced as reserved by a decoded SCI) | per UE: `m_sensedReservedCSRMap`, one entry per decoded SCI with its PSSCH RSRP, RRI, priority and retransmission flags |

In the DIT paper (notebook 15) a 1000 B packet at MCS 5 takes all ten subchannels, so WiLabV2Xsim
had one BR per TTI. In the MoReV2X QoS sweep (notebooks 12 to 14) the 20 MHz channel had two
subchannels and packets of 190 to 600 B.

## 2. Sensing-based selection (TS 38.214 §8.1.4, steps 1 to 7)

| Step | Standard | WiLabV2Xsim | MoReV2X |
|---|---|---|---|
| Selection window | [n + T1, n + T2], T1 ≤ Tproc,1, T2 ≤ PDB | [gen + T1, gen + T2] from the packet generation time; T1, T2 in ms (`T1autonomousMode`, `T2autonomousMode`); slots already past are removed for a re-evaluation ([BRreassignment3GPPautonomous.m:231](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L231)) | [n + 2 slots, n + PDB]: T1 is **fixed to 2 slots** at every numerology ([nr-v2x-ue-mac.cc:1924](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L1924)), T2 = remaining PDB |
| Sensing window | 1100 ms (100 ms for some RRIs) | NR uses **only the last allocation period**: row 1 of `sensingMatrixCV2X` ([BRreassignment3GPPautonomous.m:193](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L193)), i.e. 100 ms at a 100 ms RRI | 1100 ms (`m_sensingWindow`, [nr-v2x-ue-mac.cc:569](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L569)) |
| Exclusion of own transmission slots (half duplex, step 5) | slots where the UE transmitted, projected over the allowed RRIs | the UE's own transmission adds infinite power to its sensing map (`Ksi = Inf`, [sensedPowerCV2X.m:45](../../wilabv2xsim/WiLabV2Xsim/MainFilesCV2X/sensedPowerCV2X.m#L45)), so its own BRs of the last period are excluded | past transmissions projected over every allowed RRI (`UpdatePastTxInfo`, [nr-v2x-ue-mac.cc:1961](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L1961)) |
| Exclusion of reserved resources (step 6) | a resource reserved by a received SCI whose **RSRP** exceeds Th(p_rx, p_tx); periodic projection over the reservation and over the own Cresel periods | a BR is excluded if a decoded SCI announced it **and** the **total power received on that BR in the last period** exceeds one threshold ([BRreassignment3GPPautonomous.m:275](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L275)). One RRI for all, so the projection is "the same BR next period" | a CSR is excluded if a sensed SCI with **its own PSSCH RSRP** above Th(p_rx, p_tx) reserves it, projected Q times for RRIs shorter than T2 and over the own Cresel periods ([nr-v2x-ue-mac.cc:2129](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L2129), [nr-v2x-ue-mac.cc:3352](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L3352)) |
| Priority in the threshold | Th from `sl-ThresRSRP-List`, per priority pair | none | priority pair (TS 38.214 field p_i + (p_j − 1)·8), or one threshold when `PriorityAwareSensing` is off ([nr-v2x-ue-mac.cc:3267](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L3267)) |
| Too few left (step 7) | raise all thresholds by 3 dB until X % of the window remains | threshold × 2 (3 dB) until `ratioSelectedAutonomousMode` = 20 % ([BRreassignment3GPPautonomous.m:245](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L245), [BRreassignment3GPPautonomous.m:277](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L277)) | +3 dB until X = 20 % (or X per priority) |
| Choice | random among the remaining | random among all remaining BRs (NR: L2 off, [BRreassignment3GPPautonomous.m:319](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L319)) | random among the remaining CSRs |

**Consequences.**

* WiLabV2Xsim decides on the **total** power received on a BR, not on the RSRP of the UE that
  reserved it: two weak reservations on one BR can add up above the threshold, and a strong one
  that is not announced (no SCI decoded) does not exclude anything.
* WiLabV2Xsim's NR sensing remembers **one period**. A reservation seen 200 ms ago but missed in
  the last period (half duplex, a lost SCI) is forgotten; MoReV2X keeps it for 1.1 s and projects it.
* MoReV2X's T1 of 2 slots is 0.5 ms at numerology 2 and 2 ms at numerology 0; WiLabV2Xsim's T1 is
  set in ms (1 ms in the DIT paper).

## 3. Semi-persistent scheduling (TS 38.321 §5.22.1)

| | Standard | WiLabV2Xsim | MoReV2X |
|---|---|---|---|
| Reselection counter | uniform in [5, 15] for RRI ≥ 100 ms, scaled up for shorter RRIs | [5, 15] at ≥ 100 ms, [10, 30] at ≥ 50 ms, [25, 75] below (`findRCintervalAutonomous`) | [5, 15] at ≥ 100 ms, [5, 15] × 100/RRI below (`GetCresel`, [nr-v2x-utils.cc:180](../../src/MoReV2X/model/nr-v2x-utils.cc#L180)) |
| Counter decrement | once per transmitted TB | only when the UE actually transmits on its first BR ([BRreassignment3GPPautonomous.m:104](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L104)) | at the first resource of **every reserved period**, also when there is nothing to send ([nr-v2x-ue-mac.cc:2484](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L2484)) |
| Keep probability | `sl-ProbResourceKeep` when the counter expires | `probResKeep`, default 0.8, drawn at counter = 1 ([BRreassignment3GPPautonomous.m:118](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L118)) | `Pkeep`, default 0 ([nr-v2x-ue-mac.cc:2914](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L2914)) |
| Announcing the end of a reservation | the last transmission before reselection signals no next period | the SCI reserves the BR for the next period only if counter > 1 ([CV2XsensingProcedure.m:124](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/CV2XsensingProcedure.m#L124)) | not compared here |
| Reselection triggers | counter, latency (resource after the PDB), size (TB larger than the reservation) | counter ([BRreassignment3GPPautonomous.m:146](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L146)); a resource outside [T1, T2] of a new packet, or a retransmission placed before the first transmission ([BRreassignment3GPPautonomous.m:71](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L71)) | counter, latency, size, and latency + size (`reselectionTrigger`, nr-v2x-ue-mac.h:482) |
| Dynamic scheduling | new selection per packet | `dynamicScheduling` (counter fixed to 1) | `DynamicScheduling`, per flow in this fork |
| Blind retransmissions | up to 32 slots apart, announced in the SCI | **at most one**, within ±31 slots of the first, from the same Sa; both are periodic ([BRreassignment3GPPautonomous.m:375](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L375)) | up to 3 copies, within 32 slots of each other and inside the PDB (this fork); the SCI announces the next copy |

## 4. Re-evaluation (TS 38.214 §8.1.4, "re-evaluation check")

The standard: a resource that is selected but **not yet signalled** is checked again at least at
m − T3 (m its slot, T3 = Tproc,1); if it is no longer in Sa it is replaced, and only it
(TS 38.321 §5.22.1.2a).

| | WiLabV2Xsim | MoReV2X |
|---|---|---|
| When | **exactly 3 TTIs** before the **first transmission of a new selection** (`subframesToNextAlloc == 3` and `newDataIndicator`, [BRreassignment3GPPautonomous.m:57](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L57)). The 3 is hard-coded: right at 15 kHz (Tproc,1 = 3 slots), 0.75 ms instead of 2.25 ms at 60 kHz | at m − **Tproc,1** for the numerology: 3, 5, 9, 17 slots ([nr-v2x-utils.cc:156](../../src/MoReV2X/model/nr-v2x-utils.cc#L156)); every slot from m − Tproc,1 with `AllSlotsReEvaluation` |
| Which resources | both copies of the packet (first and retransmission) at that single instant | every resource not yet announced: the first transmission of a new SPS grant, **every** dynamic packet, and a retransmission more than 32 slots after the previous copy ([nr-v2x-ue-mac.cc:2729](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L2729)); `UMHReEvaluation` checks the retransmissions only |
| How | Sa rebuilt from the packet generation time, slots already past removed ([BRreassignment3GPPautonomous.m:220](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L220)) | `Mode2Step1` rebuilt on the remaining PDB ([nr-v2x-ue-mac.cc:3878](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L3878)) |
| What changes | the copy still in Sa is kept as the first resource; the retransmission is always drawn again ([BRreassignment3GPPautonomous.m:327](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L327)) | only the resources that left Sa are replaced (`V2XChangeResources`, trigger `ReEVALUATION`, [nr-v2x-ue-mac.cc:4032](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L4032)) |
| Too late | no check if the resource is fewer than 3 TTIs away | no check if fewer than Tproc,1 slots away |
| Off | not run | run and logged as "would have changed" without changing (counterfactual counts, notebook 12) |
| Option | `reEvalAfterEmptyResource`: re-evaluate after a reserved BR went unused | — |

## 5. Pre-emption (TS 38.214 §8.1.4, "pre-emption check")

The standard: a resource **already signalled** is checked at m − T3; it is pre-empted when a
reservation of **higher priority** (prio_RX < prio_TX, and below the `sl-PreemptionEnable` priority
threshold if set) overlaps it with an RSRP above the final threshold, and only that resource is
replaced.

| | WiLabV2Xsim | MoReV2X |
|---|---|---|
| Upstream | **not implemented** | implemented in this fork (`Preemption`, `PreemptionPriority`) |
| Added here | only for DIT (notebook 15): the SCI of a DIT first transmission marks the second resource as high priority ([CV2XsensingProcedure.m:137](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/CV2XsensingProcedure.m#L137)); 3 TTIs before the second copy the UE leaves that resource if it is no longer in Sa **and** carries the mark ([BRreassignment3GPPautonomous.m:93](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L93), [BRreassignment3GPPautonomous.m:309](../../wilabv2xsim/WiLabV2Xsim/MatFilesResourceAllocationCV2X/BRreassignment3GPPautonomous.m#L309)) | — |
| Priority condition | a single mark ("announced by a DIT first transmission"); ordinary reservations never pre-empt | prio_RX < prio_TX and below `PreemptionPriority`; priority 1 is never pre-empted ([nr-v2x-ue-mac.cc:3352](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L3352)) |
| Threshold | the Sa of a new selection (with its 3 dB steps) | the threshold after the 3 dB steps of the selection that would run now; a cheap test at the starting threshold first, the full `Mode2Step1` only if it hits ([nr-v2x-ue-mac.cc:3428](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L3428)) |
| Projection | the second resource itself | the resource and its next Cresel periods (step 6 c) against each reservation projected Q times |
| When and which | 3 TTIs before the DIT second copy | at m − Tproc,1 for every resource already announced, dispatched next to re-evaluation ([nr-v2x-ue-mac.cc:2734](../../src/MoReV2X/model/nr-v2x-ue-mac.cc#L2734)) |
| What changes | the second resource, drawn again in Sa | only the pre-empted resource (`V2XChangeResources`, trigger `PREEMPTION`) |

## 6. When each simulator checks

One packet generated at slot n, its resource at slot m. The marks show when each simulator runs
the re-evaluation (or the pre-emption check, same instant) relative to m, at 15 kHz (the DIT paper)
and 60 kHz (the MoReV2X QoS sweep). Red block: the resource at slot m; triangle: the check.

In [ ]:
TPROC1 = {0: 3, 1: 5, 2: 9, 3: 17}
fig, axes = plt.subplots(1, 2, figsize=(13, 2.8), sharey=True)
for ax, mu in zip(axes, [0, 2]):
    slot_ms = 1 / 2 ** mu
    m = 20  # resource slot
    for y, (name, before, col) in enumerate([("WiLabV2Xsim", 3, "#E69F00"), ("MoReV2X", TPROC1[mu], "#0072B2"),
                                              ("Standard, latest check", TPROC1[mu], "#52514e")]):
        ax.add_patch(Rectangle((m - 0.5, y - 0.3), 1, 0.6, color="#B2182B"))
        ax.plot([m - before], [y], marker="v", ms=11, color=col)
        ax.annotate("m − %d slots = %.2f ms before" % (before, before * slot_ms), (m - before, y), xytext=(m - before - 0.5, y + 0.28),
                    fontsize=8.5, ha="right", color="#1f1e1c")
    ax.set_yticks(range(3)); ax.set_yticklabels(["WiLabV2Xsim", "MoReV2X", "Standard (m − Tproc,1)"])
    ax.set_xlim(0, 22); ax.set_ylim(-0.6, 2.7); ax.set_xticks([0, 5, 10, 15, 20]); ax.set_xticklabels(["n", "", "", "", "m"])
    ax.set_xlabel("slots (%g ms each)" % slot_ms); ax.set_title("numerology %d (%d kHz)" % (mu, 15 * 2 ** mu), loc="left")
    ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

At 15 kHz both simulators check 3 slots before, as the standard asks. At 60 kHz WiLabV2Xsim still
checks 3 slots (0.75 ms) before, later than Tproc,1 = 9 slots (2.25 ms): it sees reservations
announced in the last 1.5 ms that a real UE could not process in time, so it would overstate what
re-evaluation can catch. Its source has a TODO saying T3 should be a parameter.

## 7. What the differences mean for results

| Difference | Effect |
|---|---|
| Total power on a BR (WiLabV2Xsim) vs RSRP of the reserving UE (MoReV2X) | WiLabV2Xsim excludes more under heavy load (powers add up) and less when an interferer's SCI is not decoded |
| One period of sensing (WiLabV2Xsim NR) vs 1.1 s (MoReV2X) | WiLabV2Xsim forgets a reservation missed in the last period, so half duplex and lost SCIs cost more collisions |
| One size, one RRI, no priority (WiLabV2Xsim) | no size or latency reselections, no projection of short RRIs, no priority thresholds or pre-emption by priority: right for the DIT paper (equal periodic packets), not for the mixed traffic of notebooks 12 to 14 |
| Re-evaluation once per new selection (WiLabV2Xsim) vs before every unannounced resource (MoReV2X) | with dynamic scheduling and aperiodic CPMs, MoReV2X re-evaluates every packet; WiLabV2Xsim would not. This is why re-evaluation helped the CPMs by 4 points in MoReV2X and does nothing for periodic single-copy traffic in both |
| T3 fixed to 3 slots (WiLabV2Xsim) vs Tproc,1 (MoReV2X) | equal at 15 kHz; at 60 kHz WiLabV2Xsim checks too late |
| Retransmission redrawn at every re-evaluation (WiLabV2Xsim) vs only the resources that left Sa (MoReV2X, as TS 38.321 asks) | WiLabV2Xsim changes more resources than the standard requires |
| Pre-emption only by the DIT mark (WiLabV2Xsim, added) vs by SL priority (MoReV2X) | DIT in WiLabV2Xsim is a clean test of the paper's idea; in MoReV2X it could be tested with real priorities, where a DIT first transmission at a high SL priority would pre-empt every lower class too |

**In short.** WiLabV2Xsim implements the SB-SPS core of TS 38.214 in a simplified, fast form
(power on a fixed grid, one period of memory, one size, no priorities) and re-evaluation once per
new selection at a fixed 3 slots; it has no pre-emption. MoReV2X follows the per-SCI, per-priority
rules of TS 38.214 and TS 38.321 for selection, re-evaluation and pre-emption, at the numerology's
Tproc,1, at a much higher cost per simulated second.